# STATE SE — One-Hot Embeddings (PBMC 46k)

Train STATE with **one-hot gene embeddings** instead of ESM-2 protein embeddings.
Same Geneformer-scale architecture (emsize=256, nhead=4, nlayers=3), pad_length=2048.

This isolates the contribution of pre-trained ESM-2 protein embeddings vs learned-from-scratch representations.

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.expanduser(
    "~/noise_scaling/modeling/Scaling-up-measurement-noise-scaling-laws/scaling_laws/src"
))

from pathlib import Path
import glob
import shutil
import subprocess
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import umap

from scaling_laws.prepare.data import Experiments, PrepareData
from scaling_laws.algo import State

In [ ]:
DATA_DIR = Path(os.path.expanduser("~/noise_scaling/data"))
DATASET = "PBMC"
SIZE = 46_415
QUALITY = 1.0
DEVICE = 4
SEED = 42

## 1. Prepare STATE data (one-hot)

Re-run `state emb preprocess` **without** `--all-embeddings` so it creates one-hot gene embeddings.
This produces a separate profile in a `state_data_onehot/` directory.

In [ ]:
base_dir = DATA_DIR / DATASET / str(SIZE) / str(QUALITY)
profile_name = f"scaling_{DATASET}_{SIZE}_{str(QUALITY).replace('.', '_')}"

state_python = Path("/home/igor/miniconda3/envs/state/bin/python")
state_package_dir = Path("/home/igor/noise_scaling/modeling/STATE/state")
state_defaults_yaml = state_package_dir / "src" / "state" / "configs" / "state-defaults.yaml"

# Output to a separate directory so we don't clobber the ESM profiles
profile_dir = base_dir / "preprocessed" / "state_data_onehot"
profile_dir.mkdir(parents=True, exist_ok=True)

marker = profile_dir / f"all_embeddings_{profile_name}.pt"
if marker.exists():
    print(f"One-hot profile already prepared: {marker}")
else:
    # Write CSV manifests (same as Experiments.prepare_state_data)
    train_base = base_dir
    val_base = DATA_DIR / DATASET / "validation" / str(QUALITY)
    test_base = DATA_DIR / DATASET / "test" / str(QUALITY)

    pd_train = PrepareData(base_dir=str(train_base))
    pd_val = PrepareData(base_dir=str(val_base))
    pd_test = PrepareData(base_dir=str(test_base))

    train_csv = pd_train.prepare_for_state(profile_name, split="train")
    pd_val.prepare_for_state(profile_name, split="val")
    pd_test.prepare_for_state(profile_name, split="test")

    val_h5ad = val_base / "preprocessed" / "preprocessed.h5ad"
    test_h5ad = test_base / "preprocessed" / "preprocessed.h5ad"
    combined_val_csv = profile_dir / "val_combined.csv"
    combined_val_csv.write_text(
        f"species,path,names\n"
        f"human,{val_h5ad},{profile_name}_val\n"
        f"human,{test_h5ad},{profile_name}_test\n"
    )

    # Copy default config
    config_path = profile_dir / "state_config.yaml"
    shutil.copy(state_defaults_yaml, config_path)

    # Run preprocess WITHOUT --all-embeddings -> creates one-hot
    cmd = [
        str(state_python), "-m", "state", "emb", "preprocess",
        "--profile-name", profile_name,
        "--train-csv", str(train_csv),
        "--val-csv", str(combined_val_csv),
        "--output-dir", str(profile_dir),
        "--config-file", str(config_path),
    ]
    print(f"Running: {' '.join(cmd)}")
    subprocess.run(cmd, cwd=str(state_package_dir), check=True)

    # Patch config: val-only (no test leakage)
    from omegaconf import OmegaConf
    cfg = OmegaConf.load(str(config_path))
    preprocessed_val_csv = Path(cfg.dataset[profile_name].val)
    val_only_out = profile_dir / f"val_only_{profile_name}.csv"
    df_val = pd.read_csv(str(preprocessed_val_csv))
    df_val = df_val[df_val["names"].str.endswith("_val")]
    df_val.to_csv(str(val_only_out), index=False)
    cfg.dataset[profile_name].val = str(val_only_out)
    cfg.dataset[profile_name].num_datasets = len(
        pd.read_csv(str(cfg.dataset[profile_name].train))
    ) + len(df_val)
    OmegaConf.save(cfg, str(config_path))
    print(f"One-hot profile saved to {profile_dir}")

In [ ]:
import torch

profile_dir = base_dir / "preprocessed" / "state_data_onehot"
profile_name = f"scaling_{DATASET}_{SIZE}_{str(QUALITY).replace('.', '_')}"

emb_file = profile_dir / f"all_embeddings_{profile_name}.pt"
emb = torch.load(emb_file, map_location="cpu", weights_only=False)
if isinstance(emb, dict):
    first_key = next(iter(emb))
    print(f"Embedding type: dict, {len(emb)} genes")
    print(f"Embedding dim: {emb[first_key].shape[0]} (should equal num genes for one-hot)")
else:
    print(f"Embedding shape: {emb.shape}")

## 2. Train (one-hot embeddings)

In [ ]:
base_dir = DATA_DIR / DATASET / str(SIZE) / str(QUALITY)

model = State(
    base_dir=str(base_dir),
    device=DEVICE,
    max_epochs=100,
    early_stopping_patience=3,
    dataset_name=DATASET,
    seed=SEED,
    pad_length=2048,
    emsize=256,
    d_hid=512,
    nhead=4,
    nlayers=3,
    output_dim=256,
    batch_size=64,
    max_lr=1e-4,
)

# Point at one-hot profile and distinct results dir
model.profile_dir = base_dir / "preprocessed" / "state_data_onehot"
model.config_path = model.profile_dir / "state_config.yaml"
model.save_folder_path = base_dir / "results" / "State_onehot"
model.model_name = "model"
model.checkpoint_dir = model.save_folder_path / "model" / "checkpoints"
model.embeddings_path = model.save_folder_path / "model" / "embeddings.csv"

print(f"Profile dir: {model.profile_dir}")
print(f"Save folder: {model.save_folder_path}")
print(f"Config: {model.config_path}")

In [ ]:
model.train()

## 3. Training / validation loss curves

In [ ]:
log_dirs = sorted(glob.glob(
    str(model.checkpoint_dir / f"state_{model.profile_name}" / "version_*")
))
metrics_file = Path(log_dirs[-1]) / "metrics.csv"
print(f"Reading: {metrics_file}")

df = pd.read_csv(metrics_file)
train_loss = df[["step", "trainer/train_loss"]].dropna()
val_loss = df[["step", "validation/val_loss"]].dropna()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_loss["step"], train_loss["trainer/train_loss"],
        label="train loss", color="blue", linewidth=1)
ax.plot(val_loss["step"], val_loss["validation/val_loss"],
        color="orange", marker="o", label="val loss", markersize=4, linewidth=1)
ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.set_title(f"STATE SE (one-hot) — PBMC {SIZE}, q={QUALITY}")
ax.set_xscale("log")
ax.set_yscale("log")
ax.legend()
ax.grid(True, alpha=0.3, which="both", linestyle="--")
fig.tight_layout()
plt.show()

## 4. Embed test set

In [ ]:
embeddings = model.embed()
print(f"Embeddings shape: {embeddings.shape}")

## 5. Compute LMI (protein_counts)

In [ ]:
model.signal_columns = ["protein_counts"]
mi_results = model.mutual_information(max_epochs=300)
print("\nLMI results:")
for signal, mi in mi_results.items():
    print(f"  {signal}: {mi:.5f}")

## 6. Compare LMI across methods

In [ ]:
results_root = base_dir / "results"
algos = [
    ("PCA",              "PCA"),
    ("RandomProjection", "RP"),
    ("SCVI",             "scVI"),
    ("Geneformer",       "Geneformer"),
    ("State",            "State (ESM-2)"),
    ("State_onehot",     "State (one-hot)"),
]

rows = []
for algo_dir, algo_label in algos:
    sig = "Y_protein_counts_1.0_geneformer" if algo_dir == "Geneformer" else "Y_protein_counts_1.0"
    mi_base = results_root / algo_dir / "model" / "MI"
    if not mi_base.exists():
        print(f"{algo_label:20s}  NOT FOUND")
        continue
    for seed_dir in sorted(mi_base.iterdir()):
        mi_file = seed_dir / sig / "lmi_mutual_information.txt"
        if mi_file.exists():
            mi = float(mi_file.read_text().strip())
            rows.append({"Algorithm": algo_label, "seed": int(seed_dir.name), "LMI": mi})
            print(f"{algo_label:20s}  seed={seed_dir.name}  LMI={mi:.5f}")

scores = pd.DataFrame(rows)
scores_agg = (
    scores.groupby("Algorithm")["LMI"]
    .agg(["mean", "std", "count"])
    .rename(columns={"mean": "mean_lmi", "std": "std_lmi", "count": "n_seeds"})
    .reset_index()
    .sort_values("mean_lmi", ascending=False)
)
scores_agg["std_lmi"] = scores_agg["std_lmi"].fillna(0)

colors = {"PCA": "#4C72B0", "RP": "#DD8452", "scVI": "#55A868",
          "Geneformer": "#C44E52", "State (ESM-2)": "#8172B3", "State (one-hot)": "#DA8BC3"}

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(
    scores_agg["Algorithm"], scores_agg["mean_lmi"],
    yerr=scores_agg["std_lmi"], capsize=4,
    color=[colors.get(a, "#999") for a in scores_agg["Algorithm"]],
    edgecolor="black", linewidth=0.5,
)
for bar, mean, std, n in zip(bars, scores_agg["mean_lmi"], scores_agg["std_lmi"], scores_agg["n_seeds"]):
    label = f"{mean:.3f}"
    if n > 1:
        label += f"\n\u00b1{std:.3f} (n={n})"
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + std + 0.02,
            label, ha="center", va="bottom", fontsize=9)
ax.set_ylabel("LMI (protein_counts)")
ax.set_title(f"PBMC {SIZE}, q={QUALITY} — ESM-2 vs one-hot embeddings")
ax.grid(axis="y", alpha=0.3)
ax.set_ylim(0, (scores_agg["mean_lmi"] + scores_agg["std_lmi"]).max() * 1.2)
plt.xticks(rotation=15, ha="right")
fig.tight_layout()
plt.show()

## 7. UMAP

In [ ]:
max_cells = 10_000
if embeddings.shape[0] > max_cells:
    rng = np.random.default_rng(42)
    idx = rng.choice(embeddings.shape[0], max_cells, replace=False)
    emb_sub = embeddings[idx]
else:
    emb_sub = embeddings

reducer = umap.UMAP(n_components=2, random_state=42, n_jobs=1)
umap_coords = reducer.fit_transform(emb_sub)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(umap_coords[:, 0], umap_coords[:, 1], color="#888", s=2, alpha=0.7)
ax.set_title(f"STATE SE (one-hot) — PBMC {SIZE}, q={QUALITY}")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
fig.tight_layout()
plt.show()